# 🦀 Day 4: File System Operations — walkdir
---

Today we explore **recursive directory traversal** and file system operations.

- **std::fs**: read_dir, metadata, file_type recap
- **std::path**: Path and PathBuf operations
- **walkdir**: Recursive traversal with WalkDir
- **Filtering**: By extension, size, modification time
- **Depth control**: max_depth(), min_depth()
- **Symlinks**: Following vs not following

In [ ]:
:dep walkdir = "2"

In [ ]:
use std::fs;
use std::path::Path;

// std::fs recap
let entries = fs::read_dir(".").expect("read_dir failed");
for entry in entries.flatten() {
    let path = entry.path();
    let meta = entry.metadata().unwrap();
    let ft = meta.file_type();
    let kind = if ft.is_dir() { "dir" } else if ft.is_file() { "file" } else { "other" };
    println!("{} ({})", path.display(), kind);
}

In [ ]:
use walkdir::WalkDir;

// Recursive traversal with walkdir
let walker = WalkDir::new(".")
    .max_depth(2)  // limit depth
    .follow_links(false)
    .into_iter()
    .filter_map(|e| e.ok());

for entry in walker {
    let path = entry.path();
    if path.is_file() {
        let ext = path.extension().and_then(|e| e.to_str()).unwrap_or("");
        println!("{} [{}]", path.display(), ext);
    }
}

In [ ]:
// Filter by extension and size
use std::collections::HashMap;

let mut by_ext: HashMap<String, (u64, usize)> = HashMap::new();

for entry in WalkDir::new(".").into_iter().filter_map(|e| e.ok()) {
    if entry.file_type().is_file() {
        let path = entry.path();
        let ext = path.extension()
            .and_then(|e| e.to_str())
            .unwrap_or("none")
            .to_string();
        let size = entry.metadata().map(|m| m.len()).unwrap_or(0);
        let (total, count) = by_ext.entry(ext).or_insert((0, 0));
        *total += size;
        *count += 1;
    }
}

for (ext, (size, count)) in by_ext {
    println!(".{}: {} files, {} bytes", ext, count, size);
}

In [ ]:
// DirEntry methods: path(), file_type(), metadata(), depth()
// min_depth(1) skips the root; max_depth(3) limits recursion

let count = WalkDir::new(".")
    .min_depth(1)
    .max_depth(2)
    .into_iter()
    .filter_map(|e| e.ok())
    .filter(|e| e.file_type().is_file())
    .count();

println!("Files at depth 1-2: {}", count);

## 📝 Exercise: Find files larger than N bytes matching a pattern

Build a finder that:
- Takes a directory, min size (bytes), and optional extension filter
- Walks recursively
- Prints paths of files larger than min size (and matching extension if given)

In [ ]:
// YOUR CODE HERE
// fn find_large(dir: &str, min_bytes: u64, ext_filter: Option<&str>) {
//     for entry in WalkDir::new(dir).into_iter().filter_map(|e| e.ok()) {
//         if entry.file_type().is_file() {
//             let size = entry.metadata().map(|m| m.len()).unwrap_or(0);
//             if size >= min_bytes {
//                 let path = entry.path();
//                 let matches = ext_filter.map_or(true, |e| path.extension().map_or(false, |x| x == e));
//                 if matches { println!("{}", path.display()); }
//             }
//         }
//     }
// }

## 🎯 Key Takeaways

1. **std::fs::read_dir** lists a single directory; **walkdir** does recursive traversal
2. **Path** and **PathBuf** handle path operations; use `extension()`, `file_name()`
3. **WalkDir** with `max_depth()`, `min_depth()`, `follow_links()`
4. **DirEntry** provides `path()`, `file_type()`, `metadata()`, `depth()`
5. Filter with `filter_map(|e| e.ok())` to skip errors, or handle them explicitly
6. Combine with metadata for size, modification time filtering

---
**Tomorrow:** Regular expressions with the regex crate